# Lab 20 - Registry Sync identity gaps

This notebook is a guided experiment for a third-party agent that is already
visible in Agent 365 through Registry Sync. It asks what can be observed and
what Agent 365 operations remain possible when the synchronized inventory record
does not contain a Blueprint ID or Entra Agent ID.

A new user should complete the notebook from top to bottom:

1. Prepare a local Python environment and an approved Microsoft Entra public-client app.
2. Run a read-only inventory pass and select one disposable or approved agent.
3. Create or reuse one Blueprint and Blueprint principal for an approved group.
4. Create or reuse an Agent Identity under that Blueprint.
5. Read, update, or explicitly attempt the first companion Agent Registration.
6. Record the result and retain objects by default.

The default example is Google Vertex AI / `Test V2`, but it is only a starting
configuration. Change it to an approved package that is visible to your account.
The current source-metadata parser is specific to the observed Google Vertex AI
shape; do not assume that another provider uses the same fields.

**Expected result:** inventory visibility, Entra identity objects, and Agent
Registration are evaluated separately. A successful read does not prove runtime
authentication or governance enforcement, and a failed registration operation is
recorded as its own result rather than retried automatically.

Run one cell at a time in an ignored `workspace` copy. Use only non-production
objects. The notebook does not create Registry Sync connectors, modify the
third-party runtime, configure credentials, or mutate the original synchronized
package. Agent Registration APIs used here are beta.

## Prerequisites

### 1. Access and safe test data

Before opening the notebook, confirm all of the following:

- You have an approved non-production Microsoft 365 tenant or an approved disposable target.
- Registry Sync has already imported at least one third-party agent. This notebook does not create a connector or trigger a sync.
- Your signed-in user is allowed to read Agent 365 package inventory.
- An app owner or tenant administrator can configure delegated Microsoft Graph permissions and grant admin consent where required.
- You will keep tenant IDs, client IDs, object IDs, tokens, endpoints, and raw responses in the ignored working copy and evidence directory.

### 2. Create or validate the Microsoft Entra login app

Prefer an existing tenant-owned app that has already been approved for this lab.
If your organization approves a dedicated app, configure it in the
[Microsoft Entra admin center](https://entra.microsoft.com/) as follows:

1. Open **Identity > Applications > App registrations**.
2. Select **New registration**.
3. Enter a descriptive name and select **Accounts in this organizational directory only**.
4. Register the app. Do not create a client secret; this notebook is a public desktop client.
5. Open **Authentication > Add a platform > Mobile and desktop applications**.
6. Add the exact system-browser redirect URI `http://localhost`. Do not add it as a Web or SPA redirect.
7. Under **Authentication > Advanced settings**, set **Allow public client flows** to **Yes**, then save.
8. On **Overview**, copy the **Application (client) ID** and **Directory (tenant) ID** for the private sign-in prompts.

The Application (client) ID is not the app registration's Object ID, a service
principal ID, a Blueprint ID, or an Agent Identity ID. `AADSTS500113` normally
means the redirect URI is missing or configured under the wrong platform.

Official references: [desktop app configuration](https://learn.microsoft.com/entra/identity-platform/scenario-desktop-app-configuration),
[register an application](https://learn.microsoft.com/entra/identity-platform/quickstart-register-app), and
[MSAL Python interactive token acquisition](https://learn.microsoft.com/entra/msal/python/getting-started/acquiring-tokens#public-clients-interactive-token-acquisition).

### 3. Configure delegated Microsoft Graph permissions

Configure the app registration as follows:

1. Open **API permissions** and select **Add a permission**.
2. Select **Microsoft Graph > Delegated permissions**. Do not choose Application permissions.
3. Search for each exact permission name below, select its checkbox, and choose **Add permissions**.
4. Repeat until the baseline read set is present. Add the write set only for an approved write run.

| Permission | Used for | Admin consent |
| --- | --- | --- |
| `User.Read` | Verify the signed-in operator | No |
| `CopilotPackages.Read.All` | List and read Agent 365 packages | No, unless tenant policy requires it |
| `AgentIdentityBlueprint.Read.All` | Read Blueprint applications | Yes |
| `AgentIdentityBlueprintPrincipal.Read.All` | Read Blueprint principals | Yes |
| `AgentIdentity.Read.All` | Read Agent Identities | Yes |
| `AgentRegistration.Read.All` | Read a known Agent Registration | No, unless tenant policy requires it |
| `AgentIdentityBlueprint.Create` | Create a Blueprint application | Yes |
| `AgentIdentityBlueprintPrincipal.Create` | Create a Blueprint principal | Yes |
| `AgentIdentity.Create.All` | Create an Agent Identity | Yes |
| `AgentRegistration.ReadWrite.All` | Create, update, or delete Agent Registrations | Yes |
| `AgentIdentity.DeleteRestore.All` | Optional Agent Identity cleanup | Yes |
| `AgentIdentityBlueprint.DeleteRestore.All` | Optional Blueprint cleanup | Yes |

The normal read-only sign-in requests the first six read permissions as one set.
When `RUN_WRITES = True`, Prep also requests all four main create/write permissions
as one set: Blueprint create, principal create, Agent Identity create, and Agent
Registration read/write. Cleanup delete permissions are requested later only when
that cleanup target is selected.

5. An administrator should select **Grant admin consent for <tenant>** after the
   required delegated permissions are present.
6. Confirm that every admin-required permission shows a green granted status for
   the intended tenant before starting the notebook.

If an Agent Identity permission is not visible
in the portal, stop and ask the tenant administrator to use the organization's
approved beta-permission process. Do not substitute a broader permission, switch
to application authentication, or modify permission grants from this notebook.

The [Microsoft Graph permissions reference](https://learn.microsoft.com/graph/permissions-reference)
is the source of truth for current permission names and consent requirements.
Consent does not replace resource ownership, directory roles, licenses, or tenant policy.

### 4. Create the local notebook environment

Install Python 3.12 or later, `uv`, and a notebook-capable editor such as VS Code
with the Jupyter extension. From the repository root, run:

```powershell
cd lab-20-registry-sync-identity-gaps\trial-2-jupyter-notebook
uv sync --extra test --index-url https://packagefeedproxy.microsoft.io/pypi/simple/
New-Item -ItemType Directory -Force -Path .\workspace | Out-Null
if (-not (Test-Path .\workspace\registry_sync_identity_walkthrough_three_parts.ipynb)) {
  Copy-Item .\registry_sync_identity_walkthrough.ipynb `
    .\workspace\registry_sync_identity_walkthrough_three_parts.ipynb
}
```

Open the workspace copy in VS Code or Jupyter and select
`.venv\Scripts\python.exe` as the kernel. Use a trusted local notebook server;
the browser and kernel must run on the same computer. Do not run multiple copies
at the same time because they share private state.

### 5. First-run checklist

- Restart the kernel and run cells in order.
- Leave every mutation switch off for the first read-only pass.
- Enter tenant and client IDs only in the private prompts.
- Complete any required MFA or Conditional Access challenge.
- Stop if the selected object is production, consent is missing, a write outcome is uncertain, or a required permission is unavailable.

## Prep - configure the run

Run the next three code cells in order before signing in:

1. `configuration-variables` selects the target package, platform, Blueprint plan, and Graph paths.
2. `configuration-switches` controls every approval, write, creation, and cleanup action.
3. `setup-code` loads imports, private state, authentication state, and the guarded Graph helper.

### Configuration variables

| Variable | What a new user should enter |
| --- | --- |
| `TARGET_NAME` | Exact display name of one approved Registry Sync package |
| `PLATFORM` | Exact `platform` value expected on that package |
| `REGISTRY_SYNC_PLATFORMS` | Platform labels independently confirmed as Registry Sync sources |
| `BLUEPRINT_GROUPS` | Approved non-production Blueprint group labels per platform |
| `SELECTED_BLUEPRINT_GROUP` | The one approved group for the selected agent |
| `GRAPH`, `PACKAGES`, `REGISTRATIONS` | Documented Microsoft Graph endpoints; do not change them for normal use |

### Switches and safe defaults

| Switch | Meaning |
| --- | --- |
| `RUN_WRITES` | Master write gate. Leave `False` for the first pass. |
| `CONFIRM_EACH_WRITE` | When `True`, require `APPLY` before each normal write. |
| `GROUPS_APPROVED` | Confirms the platform/group plan and sponsor choice. |
| `AGENT_GROUP_APPROVED` | Confirms the selected agent belongs in `SELECTED_BLUEPRINT_GROUP`. |
| `CREATE_BLUEPRINT`, `CREATE_PRINCIPAL`, `CREATE_IDENTITY` | Permit only the named object type to be created when no saved/reused object is available. |
| `CREATE_COMPANION` | Permit an explicitly confirmed first Agent Registration POST. |
| `DELETE_OBJECT` | Cleanup target; leave `None` to retain everything. |
| `CONFIRMED_NO_DEPENDENTS` | Confirms the separate dependency review required before deletion. |
| `CLEANUP_PLATFORM`, `CLEANUP_GROUP` | Optional Blueprint cleanup selector; `None` uses the selected agent's group. |

For the first run, leave every switch at its default. After the read-only pass,
change only approved switches, rerun `configuration-switches`, and rerun the
sign-in cell so the token includes any newly required write scope.

Returned IDs and write guards are stored in ignored
`evidence/trial-2-jupyter-notebook/state.json`. Do not delete or edit that file
to force a retry. A write starts by setting `pending_write`; it is cleared only
after the expected success response. If a write fails or times out, stop and
reconcile the result before running another write.

In [2]:
# Target, platform/group plan, and Graph resource paths.
TARGET_NAME = 'Test V2'
PLATFORM = 'GoogleVertexAI'
REGISTRY_SYNC_PLATFORMS = {'GoogleVertexAI'}
BLUEPRINT_GROUPS = {'GoogleVertexAI': ['test-v2-dev']}
SELECTED_BLUEPRINT_GROUP = 'test-v2-dev'

GRAPH = 'https://graph.microsoft.com'
PACKAGES = '/v1.0/copilot/admin/catalog/packages'
REGISTRATIONS = '/beta/copilot/agentRegistrations'


In [3]:
# Safety, approval, creation, and cleanup switches.
RUN_WRITES = True
CONFIRM_EACH_WRITE = False
GROUPS_APPROVED = True
AGENT_GROUP_APPROVED = True
CREATE_BLUEPRINT = False
CREATE_PRINCIPAL = False
CREATE_IDENTITY = False
CREATE_COMPANION = True

DELETE_OBJECT = None
CONFIRMED_NO_DEPENDENTS = False
CLEANUP_PLATFORM = None
CLEANUP_GROUP = None


In [4]:
import json
from datetime import datetime
from time import time
from getpass import getpass
from pathlib import Path
from urllib.parse import quote, urlsplit
import requests
import msal

folders = [Path.cwd(), *Path.cwd().parents]
candidates = [p for folder in folders for p in (folder, folder / 'lab-20-registry-sync-identity-gaps')]
LAB_ROOT = next(p for p in candidates if (p / 'identity-assignment-research.md').is_file())
PRIVATE = LAB_ROOT / 'evidence' / 'trial-2-jupyter-notebook'
PRIVATE.mkdir(parents=True, exist_ok=True)
STATE_FILE = PRIVATE / 'state.json'
state = json.loads(STATE_FILE.read_text(encoding='utf-8')) if STATE_FILE.exists() else {}
access_token = None
auth_client = None
auth_account = None
access_token_scopes = set()
access_token_expires_at = 0

def save_state():
    temporary = STATE_FILE.with_suffix('.tmp')
    temporary.write_text(json.dumps(state, indent=2), encoding='utf-8')
    temporary.replace(STATE_FILE)

def request_graph(method, path, body=None, *, record=None, expected=None):
    url = GRAPH + path if path.startswith('/') else path
    parsed = urlsplit(url)
    assert parsed.scheme == 'https' and parsed.hostname == 'graph.microsoft.com', 'Unexpected Graph host.'
    assert not parsed.username and not parsed.password and parsed.port in (None, 443), 'Unexpected Graph authority.'
    assert access_token, 'Sign in first.'
    writing = method != 'GET'
    if writing:
        if not RUN_WRITES:
            print({'method': method, 'status': 'skipped - RUN_WRITES is False'})
            return None
        assert GROUPS_APPROVED, 'Approve the configured Blueprint groups in Part 1 first.'
        assert record and record not in state.get('completed_writes', []), 'Write already recorded; read its result instead of repeating it.'
        assert not state.get('pending_write'), 'An earlier write is unresolved. Inspect private state; do not resend it.'
        if method == 'DELETE' or CONFIRM_EACH_WRITE:
            assert getpass(f'Type APPLY to approve this {method} ({record}): ') == 'APPLY', 'Not approved.'
        state['pending_write'] = {'method': method, 'path': path, 'record': record}
        save_state()
    try:
        response = requests.request(method, url, headers={'Authorization': 'Bearer ' + access_token, 'Accept': 'application/json', 'OData-Version': '4.0'}, json=body, timeout=60, allow_redirects=False)
    except requests.RequestException:
        raise RuntimeError('Network error. If this was a write, its outcome is unknown; do not repeat it.') from None
    allowed = expected or {'GET': (200,), 'POST': (201,), 'PATCH': (200,), 'DELETE': (204,)}[method]
    print({'method': method, 'http_status': response.status_code})
    assert response.status_code in allowed, f'HTTP {response.status_code}; stop and resolve the error. No automatic retry.'
    if response.status_code == 404:
        return None
    try:
        data = response.json() if response.content else {}
    except requests.exceptions.JSONDecodeError:
        raise RuntimeError('Response was not JSON. A write may have succeeded; inspect before retrying.') from None
    assert isinstance(data, dict), 'Unexpected response shape.'
    if writing:
        if method == 'POST':
            assert data.get('id'), 'Create returned no ID; its outcome needs investigation.'
            state.setdefault('created_here', {})[record] = data['id']
        state[record] = data
        state.setdefault('completed_writes', []).append(record)
        state.pop('pending_write')
        save_state()
    return data

print({'target': TARGET_NAME, 'platform': PLATFORM, 'writes_enabled': RUN_WRITES})


{'target': 'Test V2', 'platform': 'GoogleVertexAI', 'writes_enabled': True}


### Sign in and verify the operator

The first prompt asks for the **Directory (tenant) ID** and the second asks for
the **Application (client) ID** copied from the approved app registration. Both
values are saved only in the ignored state file. The browser then opens for the
approved user to sign in and complete MFA or Conditional Access.

The notebook calls `GET /me` after authentication and saves the operator Object
ID. Every later sign-in must resolve to the same operator. The same in-memory MSAL
client and token are reused across Parts 1-3; the browser should not reopen unless
a new scope, token expiry, or tenant policy requires it.

If sign-in fails, verify the tenant/client IDs, the exact `http://localhost`
desktop redirect, public-client flow, delegated permission type, and admin-consent
status. Do not print the token response. Restarting the kernel clears the token cache.

In [5]:
state['tenant_id'] = state.get('tenant_id') or getpass('Approved tenant ID: ').strip()
state['client_id'] = state.get('client_id') or getpass('Existing approved public-client ID: ').strip()
assert state['tenant_id'] and state['client_id'], 'Tenant and client are required.'
save_state()
DISCOVERY_SCOPES = ['User.Read', 'CopilotPackages.Read.All']
ENTRA_READ_SCOPES = ['User.Read', 'AgentIdentityBlueprint.Read.All', 'AgentIdentityBlueprintPrincipal.Read.All', 'AgentIdentity.Read.All']
ENTRA_CREATE_SCOPES = ['AgentIdentityBlueprint.Create', 'AgentIdentityBlueprintPrincipal.Create', 'AgentIdentity.Create.All']
REGISTRATION_READ_SCOPES = ['AgentRegistration.Read.All']
SESSION_SCOPES = DISCOVERY_SCOPES + ENTRA_READ_SCOPES + REGISTRATION_READ_SCOPES
if RUN_WRITES:
    SESSION_SCOPES += ENTRA_CREATE_SCOPES + ['AgentRegistration.ReadWrite.All']

def sign_in(scopes):
    global access_token, auth_client, auth_account, access_token_scopes, access_token_expires_at
    requested = {scope.lower() for scope in scopes}
    if access_token and requested <= access_token_scopes and time() < access_token_expires_at - 60:
        print({'signed_in': 'reused', 'permission_count': len(requested)})
        return
    if auth_client is None:
        auth_client = msal.PublicClientApplication(state['client_id'], authority='https://login.microsoftonline.com/' + state['tenant_id'], exclude_scopes=['offline_access'])
    full_scopes = [GRAPH + '/' + scope for scope in sorted(set(scopes))]
    result = auth_client.acquire_token_silent(full_scopes, account=auth_account) if auth_account else None
    mode = 'silent'
    if not result:
        mode = 'interactive'
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        options = {'scopes': full_scopes, 'timeout': 300}
        if auth_account and auth_account.get('username'):
            options['login_hint'] = auth_account['username']
        else:
            options['prompt'] = msal.Prompt.SELECT_ACCOUNT
        print('Opening Microsoft Entra sign-in in your browser. Complete sign-in there, including any required MFA or consent.')
        result = auth_client.acquire_token_interactive(**options)
    if not result.get('access_token'):
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        auth_account = None
        auth_client = None
        raise RuntimeError('Browser sign-in did not complete. Check the browser message and the approved http://localhost desktop redirect URI; token responses are not displayed.')
    access_token = result['access_token']
    granted = {scope.rsplit('/', 1)[-1].lower() for scope in result['scope'].split()} if result.get('scope') else requested.copy()
    if not requested <= granted:
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        auth_account = None
        auth_client = None
        raise RuntimeError('The token does not contain every requested permission.')
    access_token_scopes = granted
    access_token_expires_at = int(result.get('expires_on') or time() + int(result.get('expires_in', 300)))
    operator = request_graph('GET', '/v1.0/me')
    if state.get('operator_id') and state['operator_id'] != operator['id']:
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        auth_account = None
        auth_client = None
        raise RuntimeError('Operator changed; stop and review ownership.')
    claims = result.get('id_token_claims', {})
    username = claims.get('preferred_username')
    accounts = auth_client.get_accounts(username=username) if username else auth_client.get_accounts()
    selected_oid = claims.get('oid')
    auth_account = next((account for account in accounts if account.get('local_account_id') == selected_oid), accounts[0] if len(accounts) == 1 else auth_account)
    state['operator_id'] = operator['id']
    save_state()
    print({'signed_in': mode, 'permission_count': len(requested)})

sign_in(SESSION_SCOPES)


Opening Microsoft Entra sign-in in your browser. Complete sign-in there, including any required MFA or consent.
{'method': 'GET', 'http_status': 200}
{'signed_in': 'interactive', 'permission_count': 10}


### Read the visible package inventory

This cell follows pagination for `GET /v1.0/copilot/admin/catalog/packages` and
prints only the number of packages visible to the signed-in operator. It does not
write anything. "All" means all packages returned to this user, not proof of a
complete tenant inventory.

If the count is zero or the request is denied, stop here and resolve package
visibility or `CopilotPackages.Read.All`. Do not enable writes to work around a
read failure. Display names are not unique; Part 2 resolves the exact package.

In [6]:
def list_packages():
    packages, seen = [], set()
    url = GRAPH + PACKAGES
    while url:
        assert url not in seen and len(seen) < 1000, 'Repeated or excessive pagination.'
        assert urlsplit(url).path == PACKAGES, 'Unexpected pagination resource.'
        seen.add(url)
        page = request_graph('GET', url)
        packages.extend(page['value'])
        url = page.get('@odata.nextLink')
    return packages


packages = list_packages()
print({'visible_packages': len(packages)})


{'method': 'GET', 'http_status': 200}
{'visible_packages': 356}


## Part 1 - Prepare the Blueprint group

### 1.1 Review platforms in the inventory

This cell groups the already-loaded package list by its returned `platform` value
and prints counts. It makes no additional Graph request and performs no writes.
A missing platform remains visible as `(platform missing)` instead of being guessed.

A platform label is not proof that a package came from Registry Sync. Confirm the
selected third-party platform through approved connector or portal records before
adding it to `REGISTRY_SYNC_PLATFORMS`. Package Details are read only for the one
selected target in Part 2.

In [7]:
packages_by_platform = {}
for package in packages:
    platform = package.get('platform') or '(platform missing)'
    packages_by_platform.setdefault(platform, []).append(package)
for platform, members in sorted(packages_by_platform.items()):
    print({'platform': platform, 'packages': len(members)})


{'platform': 'Copilot Studio', 'packages': 11}
{'platform': 'Foundry', 'packages': 29}
{'platform': 'GoogleVertexAI', 'packages': 5}
{'platform': 'Microsoft 365 Copilot Agent Builder', 'packages': 4}
{'platform': 'Not Available', 'packages': 302}
{'platform': 'Salesforce', 'packages': 2}
{'platform': 'fabrikam', 'packages': 3}


### 1.2 Validate the platform/group plan and sponsor

Edit `REGISTRY_SYNC_PLATFORMS` and `BLUEPRINT_GROUPS` in
`configuration-variables`, then rerun that cell. Each group should represent an
approved non-production access boundary with a responsible owner. Group labels
must be nonempty, unique within the platform, and contain no colon.

Set `GROUPS_APPROVED = True` in `configuration-switches` only after reviewing the
exact plan. This approves the listed groups, not every package on the platform.
`blueprint_bindings[platform][group]` is a private state lookup for the actual
Blueprint and principal records; it is not an Entra group object.

The sponsor defaults to the signed-in operator. To use another approved sponsor,
set `state['sponsor_id']` privately before this cell. Use that user's Entra Object
ID from **Identity > Users > <user> > Overview**, not an app/client ID, tenant ID,
or placeholder. The notebook validates GUID shape but cannot prove the user is an
approved sponsor.

In [8]:
from uuid import UUID

# 1. Validate the configured platform/group plan against the listed inventory.
assert REGISTRY_SYNC_PLATFORMS <= packages_by_platform.keys(), 'A selected platform is not in this visible inventory.'
assert '(platform missing)' not in REGISTRY_SYNC_PLATFORMS, 'A missing platform cannot be selected for provisioning.'
assert BLUEPRINT_GROUPS.keys() <= REGISTRY_SYNC_PLATFORMS, 'Review each configured platform as third-party Registry Sync first.'
for platform, groups in BLUEPRINT_GROUPS.items():
    assert groups and len(groups) == len(set(groups)), 'Each platform needs distinct group labels.'
    assert all(group.strip() and ':' not in group for group in groups) and ':' not in platform, 'Use nonempty labels without colons.'
    print({'platform': platform, 'groups': groups, 'approved': GROUPS_APPROVED})

# 2. Load group bindings and preserve any Blueprint saved by an earlier single-group run.
# Conflicting mappings must be resolved instead of creating replacement objects.
bindings = state.setdefault('blueprint_bindings', {})
if state.get('blueprint'):
    legacy_platform = state.get('original_before', {}).get('platform')
    legacy_group = state.get('blueprint_group')
    assert legacy_platform and legacy_group, 'Existing Blueprint needs its original platform/group mapping; do not create a replacement.'
    existing = bindings.setdefault(legacy_platform, {}).get(legacy_group)
    if existing:
        assert state[existing['blueprint_record']]['id'] == state['blueprint']['id'], 'Existing group bindings conflict.'
    else:
        bindings[legacy_platform][legacy_group] = {'blueprint_record': 'blueprint', 'principal_record': 'principal'}

# 3. Preserve a saved sponsor; otherwise default to the signed-in operator (no API write).
sponsor_id = state.get('sponsor_id') or state['operator_id']
try:
    sponsor_object_id = UUID(sponsor_id) if isinstance(sponsor_id, str) else None
except ValueError:
    sponsor_object_id = None
if sponsor_object_id is None:
    if sponsor_id:
        print('Previously saved sponsor ID was invalid. Enter a replacement in the private prompt below.')
    try:
        sponsor_object_id = UUID(getpass('Approved sponsor user Object ID: ').strip())
    except ValueError:
        raise ValueError('Sponsor must be an Entra user Object ID (GUID), not a placeholder.') from None
state['sponsor_id'] = str(sponsor_object_id)
save_state()
print({'sponsor_id_format_valid': True, 'sponsor_saved_locally': True})


{'platform': 'GoogleVertexAI', 'groups': ['test-v2-dev'], 'approved': True}
{'sponsor_id_format_valid': True, 'sponsor_saved_locally': True}


### 1.3 Read or create the Blueprint and Blueprint principal

The cell processes only the groups listed in `BLUEPRINT_GROUPS`.

- **Reuse path:** leave `CREATE_BLUEPRINT` and `CREATE_PRINCIPAL` false. Saved IDs
  are read first; if none are saved, enter the existing Blueprint application
  Object ID and principal Object ID in the private prompts.
- **Create path:** after confirming the objects do not exist, set `RUN_WRITES`,
  `GROUPS_APPROVED`, and the required `CREATE_*` switches to true, rerun the switch
  and sign-in cells, then run this cell once.

The Blueprint application has two different identifiers: its Object `id` is used
for application GET/DELETE, while its `appId` is the Blueprint parent ID used when
creating an Agent Identity. The Blueprint principal has its own service-principal
Object ID. This notebook creates no client secret or federated credential.

A 403, 404, missing ID, or interrupted write is a stop condition. It is not proof
that a replacement object should be created.

In [9]:
from uuid import UUID

# 1. Use the top-level creation switches and sign in with the approved Entra permissions.
entra_scopes = ENTRA_READ_SCOPES.copy()
if RUN_WRITES:
    entra_scopes += ENTRA_CREATE_SCOPES
sign_in(entra_scopes)

# 2. Process only configured groups, keeping stable keys for each group's saved objects.
ready_groups = set()
for platform, groups in BLUEPRINT_GROUPS.items():
    for group in groups:
        binding = bindings.setdefault(platform, {}).setdefault(group, {
            'blueprint_record': f'blueprint:{platform}:{group}',
            'principal_record': f'principal:{platform}:{group}',
        })
        bp_record, principal_record = binding['blueprint_record'], binding['principal_record']
        save_state()
        # 3. Reuse a Blueprint by its application object ID, or explicitly create one.
        bp = state.get(bp_record)
        object_id = bp['id'] if bp else ''
        if not object_id and not CREATE_BLUEPRINT:
            object_id = getpass(f'Existing Blueprint object ID for {platform}/{group}, or blank to leave unresolved: ').strip()
        if object_id:
            bp = request_graph('GET', '/v1.0/applications/' + quote(object_id, safe='') + '/microsoft.graph.agentIdentityBlueprint')
            assert bp['id'] == object_id, 'Unexpected Blueprint object.'
        elif CREATE_BLUEPRINT:
            bp = request_graph('POST', '/v1.0/applications/microsoft.graph.agentIdentityBlueprint', {
                'displayName': f'{platform} - {group} - disposable Blueprint',
                'sponsors@odata.bind': [GRAPH + '/v1.0/users/' + str(UUID(state['sponsor_id']))],
                'owners@odata.bind': [GRAPH + '/v1.0/users/' + quote(state['operator_id'], safe='')],
            }, record=bp_record)
        principal = None
        if bp:
            # 4. Prevent different groups from sharing the same Blueprint, then save it.
            assert bp.get('appId'), 'Blueprint appId is required.'
            for platform_bindings in bindings.values():
                for other in platform_bindings.values():
                    other_bp = state.get(other['blueprint_record'])
                    assert other['blueprint_record'] == bp_record or not other_bp or other_bp['appId'] != bp['appId'], 'The same Blueprint is already bound to another group.'
            state[bp_record] = bp
            save_state()
            # 5. Find/create the principal using Blueprint appId, not its application object ID.
            app_id = str(UUID(bp['appId']))
            principal = request_graph('GET', f"/v1.0/servicePrincipals(appId='{app_id}')/microsoft.graph.agentIdentityBlueprintPrincipal", expected=(200, 404))
            if principal is None and CREATE_PRINCIPAL:
                assert not state.get(principal_record), 'A known principal is unavailable; do not recreate it.'
                principal = request_graph('POST', '/v1.0/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal', {'appId': app_id}, record=principal_record)
            if principal:
                # 6. Mark the group ready for Part 2 only after confirming a matching, enabled principal.
                assert principal.get('appId') == bp['appId'] and principal.get('accountEnabled') is not False, 'Principal mismatch or disabled principal.'
                state[principal_record] = principal
                save_state()
                ready_groups.add((platform, group))
        print({'platform': platform, 'group': group, 'blueprint_available': bool(bp), 'principal_available': bool(principal)})


{'signed_in': 'reused', 'permission_count': 7}
{'method': 'GET', 'http_status': 200}
{'method': 'GET', 'http_status': 200}
{'platform': 'GoogleVertexAI', 'group': 'test-v2-dev', 'blueprint_available': True, 'principal_available': True}


## Part 2 - Resolve the selected agent and Agent Identity

### 2.1 Read Package Details and extract the source agent ID

This cell filters the visible inventory by `TARGET_NAME` and `PLATFORM`, then
reads Package Details only for those candidates. If several packages match, it
writes an ignored `candidates.json` file and asks you to enter the approved
Package ID privately. Do not choose by display name alone.

For the current Google Vertex AI experiment, the code validates the observed
Registry Sync markers and extracts the exact provider-native source agent ID from
the JSON-encoded definition. The returned Package Details `platform` value is
authoritative for the remaining notebook. Connection ID is recorded only as sync
provenance; it is not an identity or group selector.

If the target is absent, ambiguous, missing expected source metadata, or different
from a previously saved source, stop and review the private evidence. Do not edit
state to make a different package pass. Another provider requires an independently
reviewed parser.

For local troubleshooting only, you may print `detail` in the ignored working
copy. Clear the output before sharing or committing any notebook.

In [10]:
# 1. Decode embedded definition JSON and track malformed entries instead of guessing source IDs.
def source_definitions(detail):
    definitions, invalid = [], 0
    sections = detail.get('elementDetails') or []
    if not isinstance(sections, list):
        return [], 1
    for section in sections:
        if not isinstance(section, dict):
            invalid += 1
            continue
        elements = section.get('elements') or []
        if not isinstance(elements, list):
            invalid += 1
            continue
        for element in elements:
            if not isinstance(element, dict):
                invalid += 1
                continue
            raw = element.get('definition')
            if not raw:
                continue
            if not isinstance(raw, str):
                invalid += 1
                continue
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                invalid += 1
                continue
            if isinstance(parsed, dict) and parsed.get('SourceAgentId'):
                definitions.append(parsed)
    return definitions, invalid

# 2. Require exactly one usable source definition before trusting the agent metadata.
def read_source(detail):
    definitions, invalid = source_definitions(detail)
    assert not invalid and len(definitions) == 1, 'Source metadata is missing, malformed, or ambiguous; inspect privately.'
    return definitions[0]

# 3. Inspect only the requested agent's candidates and identify originals using the observed GCP markers.
candidates = [p for p in packages if p.get('platform') == PLATFORM and p.get('displayName') == TARGET_NAME]
originals = []
for candidate in candidates:
    detail = request_graph('GET', PACKAGES + '/' + quote(candidate['id'], safe=''))
    source = read_source(detail)
    source_ids = source.get('SourceIds', {})
    if source_ids.get('ConnectionId') and source_ids.get('mac.agentRegistrationType') == 'ConnectedPlatform':
        assert source_ids.get('mac.agentRegistrationProviderType') == PLATFORM, 'Provider markers disagree.'
        assert detail['id'] == candidate['id'] and detail.get('displayName') == TARGET_NAME and detail.get('platform') == PLATFORM, 'Candidate changed.'
        originals.append(detail)
# 4. Keep candidate details private and require a manual choice if several originals match.
(PRIVATE / 'candidates.json').write_text(json.dumps(originals, indent=2), encoding='utf-8')
assert originals, f'No Registry Sync candidate named {TARGET_NAME!r} was found for {PLATFORM}. Do not create anything.'
if len(originals) > 1:
    chosen_id = getpass('Approved original Package ID from private candidates.json: ')
    originals = [d for d in originals if d['id'] == chosen_id]
assert len(originals) == 1, 'Select exactly one approved original.'
# 5. Read platform directly from the package and preserve the exact provider source ID and scope.
original = detail = originals[0]
agent_platform = detail['platform']
source = read_source(original)
source_ids = source['SourceIds']
assert source_ids.get('mac.projectId') and source_ids.get('mac.region'), 'Source scope is incomplete.'
# 6. Prevent an accidental agent switch and retain the first baseline for later comparisons.
assert not state.get('original_id') or state['original_id'] == original['id'], 'Saved original differs; reconcile first.'
assert not state.get('source_agent_id') or state['source_agent_id'] == source['SourceAgentId'], 'Saved source differs.'
state.update(agent_platform=agent_platform, original_id=original['id'], source_agent_id=source['SourceAgentId'], connection_id=source_ids['ConnectionId'])
state.setdefault('original_before', original)
save_state()
print({'original_selected': True, 'connection_present': True, 'project_present': True, 'region_present': True, 'original_identity_present': bool(original.get('agentIdentityId'))})


{'method': 'GET', 'http_status': 200}
{'original_selected': True, 'connection_present': True, 'project_present': True, 'region_present': True, 'original_identity_present': False}


### 2.2 Approve the agent's Blueprint group

Set `SELECTED_BLUEPRINT_GROUP` in `configuration-variables` to one of the groups
prepared for the package's returned platform. Set `AGENT_GROUP_APPROVED = True`
only after reviewing this exact source agent's membership, then rerun both
configuration cells before this cell.

The notebook resolves the group's saved Blueprint and principal through
`bindings[platform][group]`. The Blueprint `appId`, not its Object ID, becomes the
parent identifier for the Agent Identity. An existing saved mapping cannot be
silently changed to another group.

In [11]:
# 1. Use the group and separate membership approval configured at the top.
blueprint_group = SELECTED_BLUEPRINT_GROUP
# 2. Require a prepared group from Part 1 and reject an implicit change to an existing mapping.
assert agent_platform in BLUEPRINT_GROUPS, 'Configure this returned platform in Part 1 first.'
assert blueprint_group in BLUEPRINT_GROUPS[agent_platform], 'Choose a configured group for this platform.'
assert (agent_platform, blueprint_group) in ready_groups, 'Blueprint and principal are not ready; resolve Part 1 before continuing.'
assert not state.get('blueprint_group') or state['blueprint_group'] == blueprint_group, 'An existing agent/group mapping requires a separate migration decision.'
# 3. Resolve the saved Blueprint/principal using package platform plus the chosen group, not connection ID.
binding = bindings[agent_platform][blueprint_group]
blueprint_record = binding['blueprint_record']
blueprint = state.get(blueprint_record)
principal = state.get(binding['principal_record'])
# 4. Save the selected mapping locally; this step creates no Entra objects.
state['blueprint_group'] = blueprint_group
if blueprint:
    state['blueprint'] = blueprint
if principal:
    state['principal'] = principal
save_state()
print({'platform': agent_platform, 'group': blueprint_group, 'membership_approved': AGENT_GROUP_APPROVED, 'blueprint_available': bool(blueprint), 'principal_available': bool(principal)})


{'platform': 'GoogleVertexAI', 'group': 'test-v2-dev', 'membership_approved': True, 'blueprint_available': True, 'principal_available': True}


### 2.3 Read or create the Agent Identity

- **Reuse path:** leave `CREATE_IDENTITY = False`. The notebook reads a saved
  identity first or privately asks for an existing Agent Identity Object ID.
- **Create path:** confirm no identity already exists, then set `RUN_WRITES`,
  `GROUPS_APPROVED`, `AGENT_GROUP_APPROVED`, and `CREATE_IDENTITY` to true. Rerun
  the switches and sign-in cells before running this cell once.

The create request supplies the Blueprint `appId` as the parent and the approved
sponsor. It does not accept the provider source agent ID, so creating the Entra
object does not by itself bind it to the third-party runtime.

The cell reads the Blueprint, principal, and Agent Identity back and verifies their
identifier relationship. That proves the Entra parent relationship only; it does
not prove runtime authentication, token exchange, or governance enforcement.

In [12]:
# 1. Use the top-level creation choice, preferring any identity already saved for this agent.
agent_identity = state.get('agent_identity')
if blueprint and principal:
    # 2. Read a known identity; ask for its ID only when reuse mode has no saved value.
    identity_id = agent_identity['id'] if agent_identity else ''
    if not identity_id and not CREATE_IDENTITY:
        identity_id = getpass(f'Known {TARGET_NAME} Agent Identity ID, or blank to leave unresolved: ').strip()
    if identity_id:
        agent_identity = request_graph('GET', '/v1.0/servicePrincipals/' + quote(identity_id, safe='') + '/microsoft.graph.agentIdentity')
        assert agent_identity['id'] == identity_id, 'Unexpected identity ID.'
    elif CREATE_IDENTITY:
        # 3. Create only an approved new identity under the Blueprint appId, not its application object ID.
        assert AGENT_GROUP_APPROVED, 'Approve this agent/group membership first.'
        assert not original.get('agentIdentityId'), 'Original already has an identity; reassess before creating one.'
        agent_identity = request_graph('POST', '/v1.0/servicePrincipals/microsoft.graph.agentIdentity', {
            'displayName': f'{TARGET_NAME} - disposable Agent Identity',
            'agentIdentityBlueprintId': blueprint['appId'],
            'sponsors@odata.bind': [GRAPH + '/v1.0/users/' + str(UUID(state['sponsor_id']))],
            'owners@odata.bind': [GRAPH + '/v1.0/users/' + quote(state['operator_id'], safe='')],
        }, record='agent_identity')
    if agent_identity:
        # 4. Confirm the returned identity's parent and type before saving its actual ID.
        assert agent_identity.get('agentIdentityBlueprintId') == blueprint['appId'], 'Identity has a different Blueprint parent.'
        assert agent_identity.get('servicePrincipalType') == 'ServiceIdentity', 'Unexpected identity type.'
        state['agent_identity'] = agent_identity
        save_state()
print({'agent_identity_available': bool(agent_identity)})

# 5. Re-read the directory objects to confirm their relationship, not runtime authentication.
identity_verified = False
if blueprint and principal and agent_identity:
    checked_blueprint = request_graph('GET', '/v1.0/applications/' + quote(blueprint['id'], safe='') + '/microsoft.graph.agentIdentityBlueprint')
    checked_identity = request_graph('GET', '/v1.0/servicePrincipals/' + quote(agent_identity['id'], safe='') + '/microsoft.graph.agentIdentity')
    assert checked_blueprint['id'] == blueprint['id'] and checked_blueprint['appId'] == blueprint['appId'], 'Blueprint changed.'
    assert checked_identity['id'] == agent_identity['id'] and checked_identity.get('servicePrincipalType') == 'ServiceIdentity', 'Identity changed.'
    assert checked_identity['agentIdentityBlueprintId'] == blueprint['appId'], 'Parent does not match.'
    identity_verified = True
print({'entra_relationship_verified': identity_verified})


{'method': 'GET', 'http_status': 200}
{'agent_identity_available': True}
{'method': 'GET', 'http_status': 200}
{'method': 'GET', 'http_status': 200}
{'entra_relationship_verified': True}


## Part 3 - Test the Agent Registration boundary

### 3.1 Choose reuse or first creation

Use one of two explicit modes:

- **Known registration:** leave `CREATE_COMPANION = False`. The notebook uses a
  saved Registration ID or asks for one privately, then verifies source, platform,
  ownership, and the absence of `managedByAppId` before accepting it.
- **Confirmed first creation:** set `CREATE_COMPANION = True` only when the team has
  established that no companion registration was previously created for this exact
  source. Cell 3.2 still requires the typed confirmation `FIRST`.

A Registration ID is not the original Registry Sync Package ID. The reviewed API
provides no documented list-by-source lookup. A missing ID, 403, or 404 is therefore
unavailable evidence, not permission to POST a duplicate. A saved registration is
always read and reused even if creation is enabled.

In [13]:
# 1. Use the top-level registration choice, then sign in for the required permissions.
association_scopes = DISCOVERY_SCOPES + ENTRA_READ_SCOPES + REGISTRATION_READ_SCOPES
if RUN_WRITES:
    association_scopes += ['AgentRegistration.ReadWrite.All']
sign_in(association_scopes)
# 2. Prefer the saved Registration ID; prompt only when reuse mode needs one.
registration = state.get('registration')
registration_id = registration['id'] if registration else ''
if not registration_id and not CREATE_COMPANION:
    registration_id = getpass(f'Known {TARGET_NAME} companion Registration ID, or blank if unresolved: ').strip()
if registration_id:
    # 3. Read by Registration ID, never the original Package ID; failed reads do not trigger creation.
    assert registration_id != state['original_id'], 'Do not use the original Package ID as a Registration ID.'
    registration = request_graph('GET', REGISTRATIONS + '/' + quote(registration_id, safe=''))
    # 4. Check source, platform, and ownership before accepting and saving this companion.
    assert registration['id'] == registration_id and registration.get('sourceAgentId') == state['source_agent_id'], 'Companion/source mismatch.'
    assert registration.get('originatingStore') == agent_platform, 'Wrong source platform.'
    assert state['operator_id'] in registration.get('ownerIds', []) and not registration.get('managedByAppId'), 'Expected the approved user-owned companion.'
    state['registration'] = registration
    save_state()
print({'known_companion_available': bool(registration)})


{'signed_in': 'reused', 'permission_count': 7}
{'known_companion_available': False}


### 3.2 Update the known registration or attempt the first companion

Before any write, the cell re-reads the original package and the actual Entra
objects. The identity links are the Blueprint `appId` and Agent Identity Object
`id`; the exact source agent ID comes from Part 2.

- For a known registration, the notebook PATCHes only missing identity fields. It
  stops rather than overwriting a conflicting identity, source, platform, or owner.
- For a confirmed first companion, the POST also includes display name, platform,
  creator/owner, and the original provider timestamps. Do not add `managedByAppId`.

Expected success is `200` for PATCH or `201` for POST. In the current experiment,
a POST can instead return HTTP 500 with an embedded permission-denial message even
when delegated scope and consent are present. Record that as **permission-denied**;
do not broaden permissions, delete/recreate objects, or retry automatically.

A failed or interrupted write intentionally leaves `pending_write` in private state.
Stop at that point. Preserve the response evidence and do not continue to 3.3 until
the write outcome has been reconciled.

In [14]:
# 1. Require verified Entra objects and approved agent membership before associating identities.
if identity_verified:
    if RUN_WRITES:
        assert AGENT_GROUP_APPROVED, 'Approve this agent/group membership first.'
    # 2. Re-read the original and stop if its identity status changed; build links from actual Entra IDs.
    current_original = request_graph('GET', PACKAGES + '/' + quote(state['original_id'], safe=''))
    assert not current_original.get('agentIdentityId'), 'Original now has an identity; reassess native association.'
    identity_links = {'agentIdentityBlueprintId': blueprint['appId'], 'agentIdentityId': agent_identity['id']}
    if registration:
        # 3. Recheck the companion and PATCH missing links only; do not overwrite conflicting associations.
        current = request_graph('GET', REGISTRATIONS + '/' + quote(registration['id'], safe=''))
        assert current.get('sourceAgentId') == state['source_agent_id'] and current.get('originatingStore') == agent_platform, 'Source changed.'
        assert state['operator_id'] in current.get('ownerIds', []) and not current.get('managedByAppId'), 'Ownership changed.'
        assert all(current.get(k) in (None, '', v) for k, v in identity_links.items()), 'Different identity links already exist; do not overwrite.'
        if all(current.get(k) == v for k, v in identity_links.items()):
            print({'association': 'already stored; no write needed'})
        else:
            association_result = request_graph('PATCH', REGISTRATIONS + '/' + quote(registration['id'], safe=''), identity_links, record='association_update')
    elif CREATE_COMPANION:
        # 4. Confirm first creation and preserve source timestamps; never POST merely because an ID was lost.
        assert getpass('Type FIRST to confirm a companion has never been created for this source: ') == 'FIRST', 'Creation not confirmed.'
        for timestamp in (source.get('CreatedDateTime'), source.get('LastModifiedDateTime')):
            assert isinstance(timestamp, str) and datetime.fromisoformat(timestamp).tzinfo is not None, 'Keep valid original ISO timestamps.'
        registration = request_graph('POST', REGISTRATIONS, {
            'displayName': f'{TARGET_NAME} - identity companion',
            'description': 'Disposable companion; original Registry Sync record retained',
            'createdBy': state['operator_id'], 'ownerIds': [state['operator_id']],
            'sourceAgentId': state['source_agent_id'], 'originatingStore': agent_platform,
            'sourceCreatedDateTime': source['CreatedDateTime'],
            'sourceLastModifiedDateTime': source['LastModifiedDateTime'],
            **identity_links,
        }, record='registration')
    else:
        print({'association': 'blocked - known companion required'})
else:
    print({'association': 'blocked - actual verified Entra objects required'})


{'method': 'GET', 'http_status': 200}


AssertionError: An earlier write is unresolved. Inspect private state; do not resend it.

In [ ]:
# Uncomment to use this code snippet only if to debug the http request that would be sent to create the companion registration. 
# It reconstructs the request and saves it to a private file for inspection, without actually sending it.
# It will populate the fields inside trial-2-jupyter-notebook/registration-identity-replay-template.http
# and generated evidence/trial-2-jupyter-notebook/registration-create-replay.http.
#
# Then next step you can run the request inside evidence/trial-2-jupyter-notebook/registration-create-replay.http.


# assert access_token and identity_verified and CREATE_COMPANION, 'Current authenticated creation context is required.'
# assert state.get('pending_write', {}).get('record') == 'registration', 'Expected the unresolved registration marker.'
# for timestamp in (source.get('CreatedDateTime'), source.get('LastModifiedDateTime')):
#     assert isinstance(timestamp, str) and datetime.fromisoformat(timestamp).tzinfo is not None, 'Invalid source timestamp.'
# replay_body = {
#     'displayName': f'{TARGET_NAME} - identity companion',
#     'description': 'Disposable companion; original Registry Sync record retained',
#     'createdBy': state['operator_id'],
#     'ownerIds': [state['operator_id']],
#     'sourceAgentId': state['source_agent_id'],
#     'originatingStore': agent_platform,
#     'sourceCreatedDateTime': source['CreatedDateTime'],
#     'sourceLastModifiedDateTime': source['LastModifiedDateTime'],
#     'agentIdentityBlueprintId': blueprint['appId'],
#     'agentIdentityId': agent_identity['id'],
# }
# req = requests.Request(
#     'POST',
#     GRAPH + REGISTRATIONS,
#     headers={
#         'Authorization': 'Bearer ' + access_token,
#         'Accept': 'application/json',
#         'OData-Version': '4.0',
#     },
#     json=replay_body,
# ).prepare()
# body = req.body.decode('utf-8') if isinstance(req.body, bytes) else req.body
# text = (LAB_ROOT / 'trial-2-jupyter-notebook' / 'registration-identity-replay-template.http').read_text(encoding='utf-8')
# captured = {
#     'capturedAuthorization': req.headers['Authorization'],
#     'capturedContentType': req.headers['Content-Type'],
#     'capturedAccept': req.headers['Accept'],
#     'capturedODataVersion': req.headers['OData-Version'],
#     'capturedRequestBody': body,
# }
# for name, value in captured.items():
#     marker = '{{' + name + '}}'
#     assert text.count(marker) == 1, 'Unexpected replay template.'
#     text = text.replace(marker, value)
# destination = PRIVATE / 'registration-create-replay.http'
# assert not destination.exists(), 'A private replay file already exists; inspect it instead of overwriting it.'
# with destination.open('x', encoding='utf-8', newline='\n') as output:
#     output.write(text)
# print({'private_replay_file_created': True, 'request_sent': False, 'source': 'reconstructed current context'})

{'private_replay_file_created': True, 'request_sent': False, 'source': 'reconstructed current context'}


### 3.3 Read back and classify the result

Run this section only after 3.2 completed successfully or no write was required.
It compares the registration's stored identity links with the actual Entra objects,
then separately reads the original and companion packages.

Classify each observation precisely: supported, unsupported, unavailable,
permission-denied, or inconclusive. A matching identifier string proves only that
the field was stored. It does not prove the identity was resolved, the original
Registry Sync package was enriched, or the provider runtime can authenticate with it.

For the one observed API-created companion, Registration ID was also a candidate
Package ID. Do not treat that as a general contract. Saved successful writes are
not repeated.

In [ ]:
# 1. Compare the registration's stored links with the actual Entra IDs; matching strings are not resolution proof.
links_match = False
if registration and blueprint and agent_identity:
    checked_registration = request_graph('GET', REGISTRATIONS + '/' + quote(registration['id'], safe=''))
    assert checked_registration['id'] == registration['id'] and checked_registration.get('sourceAgentId') == state['source_agent_id'], 'Unexpected registration.'
    links_match = checked_registration.get('agentIdentityBlueprintId') == blueprint['appId'] and checked_registration.get('agentIdentityId') == agent_identity['id']
    state['registration'] = checked_registration
    save_state()
print({'registration_identity_fields_match': links_match, 'semantic_resolution_proven': False})

# 2. Refresh inventory and compare the original's identity field with the saved pre-write baseline.
packages_after = list_packages()
original_after = request_graph('GET', PACKAGES + '/' + quote(state['original_id'], safe=''))
assert read_source(original_after)['SourceAgentId'] == state['source_agent_id'], 'Original source changed.'
original_identity_unchanged = original_after.get('agentIdentityId') == state['original_before'].get('agentIdentityId')
# 3. Try Registration ID as a Package ID only for this known companion, then confirm its source.
companion_package = None
if registration:
    companion_package = request_graph('GET', PACKAGES + '/' + quote(registration['id'], safe=''), expected=(200, 404))
    if companion_package:
        assert companion_package['id'] == registration['id'] and companion_package['id'] != state['original_id'], 'Unexpected package correlation.'
        assert read_source(companion_package)['SourceAgentId'] == state['source_agent_id'], 'Companion source mismatch.'
# 4. Save comparison details privately and report only observed field-level results.
state['original_after'] = original_after
state['companion_package'] = companion_package
save_state()
print({'original_identity_unchanged': original_identity_unchanged, 'companion_package_available': bool(companion_package), 'companion_identity_field_matches': bool(companion_package and agent_identity and companion_package.get('agentIdentityId') == agent_identity['id']), 'semantic_association': 'inconclusive'})

# 5. Stop on unresolved writes; otherwise read whether another association write would be necessary.
assert not state.get('pending_write'), 'A write outcome remains unknown; stop and reconcile it.'
no_write_needed = False
if registration and identity_verified:
    latest = request_graph('GET', REGISTRATIONS + '/' + quote(registration['id'], safe=''))
    no_write_needed = latest.get('agentIdentityBlueprintId') == blueprint['appId'] and latest.get('agentIdentityId') == agent_identity['id'] and latest.get('sourceAgentId') == state['source_agent_id']
print({'association_write_needed': not no_write_needed, 'original_identity_unchanged': original_identity_unchanged, 'subsequent_sync_observed': False})

# 6. Directory and registration metadata do not demonstrate provider-runtime integration.
print({'runtime_binding': 'not tested', 'provider_runtime_changed': False})


## Cleanup and limits

The safe default is `DELETE_OBJECT = None`, which retains every object. The
notebook never deletes the provider agent, Registry Sync connector, or original
synchronized package.

Cleanup is a separate approved operation. Select only one of `registration`,
`agent_identity`, or `blueprint`; set `RUN_WRITES = True`; confirm that no dependent
object remains; then set `CONFIRMED_NO_DEPENDENTS = True`. Deletion always requires
typing `APPLY`. The notebook refuses to delete reused objects and only accepts an
object recorded in `created_here` by this notebook.

Delete children before parents. A retained registration blocks Agent Identity
deletion, and a retained identity or registration blocks Blueprint deletion.
Blueprint deletion can cascade to its principal, so verify the result separately
in Microsoft Entra admin center. `CLEANUP_PLATFORM` and `CLEANUP_GROUP` can select
another newly created Part 1 group; leave both `None` for the selected agent's group.

Keep `state.json` and captured evidence while any ID or write outcome is still
needed. When finished, clear notebook outputs, restart or stop the kernel, and
remove only the exact no-longer-needed local files. Do not delete the whole evidence
or workspace directory, and never commit the working notebook or raw responses.

In [ ]:
cleanup_platform = CLEANUP_PLATFORM or agent_platform
cleanup_group = CLEANUP_GROUP or blueprint_group
if DELETE_OBJECT is None:
    print({'cleanup': 'retained - no delete requested'})
else:
    assert RUN_WRITES and CONFIRMED_NO_DEPENDENTS, 'Deletion and dependency review must be explicitly approved.'
    assert DELETE_OBJECT in ('registration', 'agent_identity', 'blueprint'), 'Unsupported cleanup target.'
    record_key = bindings[cleanup_platform][cleanup_group]['blueprint_record'] if DELETE_OBJECT == 'blueprint' else DELETE_OBJECT
    target = state[record_key]
    assert state.get('created_here', {}).get(record_key) == target['id'], 'Do not delete reused or pre-existing objects.'
    deleted = state.setdefault('confirmed_deleted', [])
    assert record_key not in deleted, 'Deletion already confirmed.'
    if DELETE_OBJECT == 'agent_identity':
        assert not state.get('registration') or 'registration' in deleted, 'The companion is still retained.'
    if DELETE_OBJECT == 'blueprint':
        identity_depends = state.get('agent_identity', {}).get('agentIdentityBlueprintId') == target['appId']
        assert not identity_depends or 'agent_identity' in deleted, 'The child identity is still retained.'
        registration_depends = state.get('registration', {}).get('agentIdentityBlueprintId') == target['appId']
        assert not registration_depends or 'registration' in deleted, 'The companion is still retained.'
    routes = {
        'registration': (REGISTRATIONS + '/' + quote(target['id'], safe=''), 'AgentRegistration.ReadWrite.All'),
        'agent_identity': ('/v1.0/servicePrincipals/' + quote(target['id'], safe='') + '/microsoft.graph.agentIdentity', 'AgentIdentity.DeleteRestore.All'),
        'blueprint': ('/v1.0/applications/' + quote(target['id'], safe='') + '/microsoft.graph.agentIdentityBlueprint', 'AgentIdentityBlueprint.DeleteRestore.All'),
    }
    delete_path, permission = routes[DELETE_OBJECT]
    sign_in(association_scopes + [permission])
    current_original = request_graph('GET', PACKAGES + '/' + quote(state['original_id'], safe=''))
    assert not current_original.get('agentIdentityId'), 'Original may now depend on the identity; do not delete it.'
    deletion_record = 'deleted_' + record_key
    if deletion_record not in state.get('completed_writes', []):
        deletion_result = request_graph('DELETE', delete_path, record=deletion_record)
    remaining = request_graph('GET', delete_path, expected=(200, 404))
    assert remaining is None, 'Object is still addressable; stop further cleanup.'
    deleted.append(record_key)
    save_state()
    print({'cleanup': 'selected object is no longer addressable', 'cascade': 'check separately in the portal'})
access_token = None
access_token_scopes = set()
access_token_expires_at = 0
auth_account = None
auth_client = None
print({'token_reference_cleared': True, 'next': 'restart or stop the kernel'})
